In [1]:
%matplotlib inline
import sys, os, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

sys.path.insert(0, '/Users/yotameviatar/vs_code')

from HR import hr_config as config
from HR import hr_processor as processor

warnings.filterwarnings('ignore', category=RuntimeWarning)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True,
                     'grid.alpha': 0.35, 'grid.linestyle': '--'})
print('Setup complete')

Setup complete


In [ ]:
# ── Configuration overrides ───────────────────────────────────────────────────
SUBJECT_FILTER       = []
ANAL_TMIN            = -5.0
ANAL_TMAX            =  10.0
BASELINE_TMIN        = -6.0
BASELINE_TMAX        =  0.0
BASELINE_METHOD      = 'mean'
SCORE_TMIN           =  2.0
SCORE_TMAX           =  6.0
HR_MIN_BPM           =  30
HR_MAX_BPM           =  200
HR_Z_SCORE_THRESHOLD =  3.0
SCORE_MAX_PCT        =  50.0

# ── Artifact cleaning ─────────────────────────────────────────────────────────
FLAT_MIN_SEC      = 8.0    # seconds frozen = probe dropout (normal blocks ≤ 7s)
SPIKE_THRESH_BPM  = 20.0   # BPM deviation from ±3-block local median = impulse spike

# ── Artifact coverage gates (baseline and score independently) ────────────────
FLAT_BASELINE_TOTAL_MAX      = 0.50   # reject if >50% of baseline was artifact
FLAT_BASELINE_CONTIGUOUS_MAX = 0.40   # reject if longest single gap > 40% of baseline
FLAT_SCORE_TOTAL_MAX         = 0.50   # reject if >50% of score window was artifact
FLAT_SCORE_CONTIGUOUS_MAX    = 0.40   # reject if longest single gap > 40% of score

# ── Plot / outlier ────────────────────────────────────────────────────────────
SHOW_SUBJECT_PLOTS     = False
IQR_OUTLIER_MULTIPLIER = 2.0

In [ ]:
# ── Load cache & apply config overrides ──────────────────────────────────────
CACHE_FILE = os.path.join(config.OUTPUT_DIR, '_cache', 'hr_cache.pkl')
with open(CACHE_FILE, 'rb') as f:
    _cache = pickle.load(f)

# Handle both cache formats: new ("sessions") and legacy ("trials")
if "sessions" in _cache:
    _raw = _cache["sessions"]
elif "trials" in _cache:
    _raw = _cache["trials"]   # processor handles old format transparently
else:
    raise RuntimeError(f"Unknown cache format: {list(_cache.keys())}")

_sessions_cache = {s: v for s, v in _raw.items()
                   if not SUBJECT_FILTER or s in SUBJECT_FILTER}

config.ANAL_TMIN                    = ANAL_TMIN
config.ANAL_TMAX                    = ANAL_TMAX
config.BASELINE_TMIN                = BASELINE_TMIN
config.BASELINE_TMAX                = BASELINE_TMAX
config.BASELINE_METHOD              = BASELINE_METHOD
config.SCORE_TMIN                   = SCORE_TMIN
config.SCORE_TMAX                   = SCORE_TMAX
config.HR_MIN_BPM                   = HR_MIN_BPM
config.HR_MAX_BPM                   = HR_MAX_BPM
config.HR_Z_SCORE_THRESHOLD         = HR_Z_SCORE_THRESHOLD
config.SCORE_MAX_PCT                = SCORE_MAX_PCT
config.SCORE_MIN_PCT                = None
config.FLAT_MIN_SEC                 = FLAT_MIN_SEC
config.SPIKE_THRESH_BPM             = SPIKE_THRESH_BPM
config.FLAT_BASELINE_TOTAL_MAX      = FLAT_BASELINE_TOTAL_MAX
config.FLAT_BASELINE_CONTIGUOUS_MAX = FLAT_BASELINE_CONTIGUOUS_MAX
config.FLAT_SCORE_TOTAL_MAX         = FLAT_SCORE_TOTAL_MAX
config.FLAT_SCORE_CONTIGUOUS_MAX    = FLAT_SCORE_CONTIGUOUS_MAX

trials_data, _traces = processor.apply_analysis_params(_sessions_cache, config)

print(f'Loaded {len(trials_data)} subjects')
for subj, sessions in sorted(trials_data.items()):
    for sk, trials in sessions.items():
        n_rej = sum(t['rejected'] for t in trials)
        n_art = sum(t.get('any_artifact', False) for t in trials)
        art_note = f', {n_art} cleaned' if n_art else ''
        print(f'  {subj} {sk}: {len(trials)} trials, {n_rej} rejected{art_note}')
        for i, t in enumerate(trials):
            if t['rejected']:
                print(f'      trial {i+1:02d}: {t.get("rejection_reason", "")}')

In [3]:
# ── Helpers ───────────────────────────────────────────────────────────────────
def _pct(epoch, baseline):
    if np.isnan(baseline) or baseline == 0:
        return epoch - baseline
    return (epoch - baseline) / baseline * 100

def _sig(p):
    return '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))

def _bracket(ax, x1, x2, y_top, p, label):
    h = (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.015
    ax.plot([x1, x1, x2, x2], [y_top, y_top+h, y_top+h, y_top], lw=1.2, color='k')
    ax.text((x1+x2)/2, y_top+h*1.5, f'{_sig(p)}  {label}  p={p:.3f}',
            ha='center', va='bottom', fontsize=9, fontweight='bold')

def _iqr_filter_paired(scores_a, scores_b, subjects, multiplier=2.0):
    """Remove paired outliers by IQR fence on the (a − b) difference."""
    if len(scores_a) < 4:
        return list(scores_a), list(scores_b), list(subjects), 0
    diffs = np.array(scores_a) - np.array(scores_b)
    q1, q3 = np.percentile(diffs, 25), np.percentile(diffs, 75)
    iqr = q3 - q1
    ok = (diffs >= q1 - multiplier * iqr) & (diffs <= q3 + multiplier * iqr)
    n_removed = int(np.sum(~ok))
    return ([s for s, v in zip(scores_a, ok) if v],
            [s for s, v in zip(scores_b, ok) if v],
            [s for s, v in zip(subjects, ok) if v],
            n_removed)

YLABEL = 'HR % change from baseline'

In [ ]:
# ── Review trials — per-subject grid, shared y-axis ──────────────────────────
# Self-contained: only needs cell-0 (imports) to have run.
REVIEW_SUBJECT_FILTER        = []     # [] = all, e.g. ['AB22']
REVIEW_FLAT_MIN_SEC          = 8.0
REVIEW_SPIKE_THRESH_BPM      = 20.0
REVIEW_BL_TOTAL_MAX          = 0.50
REVIEW_BL_CONTIGUOUS_MAX     = 0.40
REVIEW_SC_TOTAL_MAX          = 0.50
REVIEW_SC_CONTIGUOUS_MAX     = 0.40
N_COLS                       = 6

import types as _types
_rv_cache_file = os.path.join(config.OUTPUT_DIR, '_cache', 'hr_cache.pkl')
with open(_rv_cache_file, 'rb') as _f:
    _rv_cache = pickle.load(_f)

# Handle both cache formats
if "sessions" in _rv_cache:
    _rv_raw = _rv_cache["sessions"]
elif "trials" in _rv_cache:
    _rv_raw = _rv_cache["trials"]
else:
    raise RuntimeError(f"Unknown cache format: {list(_rv_cache.keys())}")

_rv_sessions = {s: v for s, v in _rv_raw.items()
                if not REVIEW_SUBJECT_FILTER or s in REVIEW_SUBJECT_FILTER}

_rv_cfg = _types.SimpleNamespace(
    **{k: getattr(config, k) for k in dir(config) if not k.startswith('_')}
)
_rv_cfg.FLAT_MIN_SEC                 = REVIEW_FLAT_MIN_SEC
_rv_cfg.SPIKE_THRESH_BPM             = REVIEW_SPIKE_THRESH_BPM
_rv_cfg.FLAT_BASELINE_TOTAL_MAX      = REVIEW_BL_TOTAL_MAX
_rv_cfg.FLAT_BASELINE_CONTIGUOUS_MAX = REVIEW_BL_CONTIGUOUS_MAX
_rv_cfg.FLAT_SCORE_TOTAL_MAX         = REVIEW_SC_TOTAL_MAX
_rv_cfg.FLAT_SCORE_CONTIGUOUS_MAX    = REVIEW_SC_CONTIGUOUS_MAX

_rv_data, _ = processor.apply_analysis_params(_rv_sessions, _rv_cfg)

for subj in sorted(_rv_data):
    sessions = _rv_data[subj]
    _accepted = [t['epoch_wide'] for sk in ['eve','mor']
                 for t in sessions.get(sk,[]) if not t['rejected']]
    if _accepted:
        _flat = np.concatenate(_accepted)
        _ylo, _yhi = np.percentile(_flat, 2), np.percentile(_flat, 98)
        _ylim = (_ylo - max((_yhi-_ylo)*0.20, 2.0), _yhi + max((_yhi-_ylo)*0.20, 2.0))
    else:
        _ylim = (40, 120)

    for sess_key in ['eve', 'mor']:
        trials = sessions.get(sess_key, [])
        if not trials:
            continue
        n      = len(trials)
        n_rows = int(np.ceil(n / N_COLS))
        fig, axes = plt.subplots(n_rows, N_COLS,
                                  figsize=(N_COLS * 3.2, n_rows * 2.8), sharey=True)
        axes_flat = np.array(axes).flatten()
        n_rej = sum(t['rejected'] for t in trials)
        n_art = sum(t.get('any_artifact', False) for t in trials)
        fig.suptitle(
            f'{subj} — {config.SESSION_MAP[sess_key]["label"]}  '
            f'({n} trials, {n_rej} rejected'
            + (f', {n_art} cleaned' if n_art else '') + ')',
            fontsize=12, fontweight='bold')

        for i, t in enumerate(trials):
            ax = axes_flat[i]
            tw = t['times_wide']; ep = t['epoch_wide']
            was_cleaned = t.get('any_artifact', False)
            prefix = '~' if was_cleaned and not t['rejected'] else ''
            if t['rejected']:
                color = config.REJ_COLOR
                title = f'T{i+1} REJ\n{t.get("rejection_reason","")[:22]}'; lw = 0.8
            elif t['label'] == 1:
                color = config.NEG_COLOR; sc = t['score']
                title = f'{prefix}T{i+1} Neg  {sc:.1f}%' if sc and not np.isnan(sc) else f'{prefix}T{i+1} Neg --'; lw = 1.0
            else:
                color = config.NEU_COLOR; sc = t['score']
                title = f'{prefix}T{i+1} Neu  {sc:.1f}%' if sc and not np.isnan(sc) else f'{prefix}T{i+1} Neu --'; lw = 1.0
            ax.plot(tw, ep, color=color, linewidth=lw, alpha=0.9)
            _art = t.get('artifact_mask')
            if was_cleaned and _art is not None and np.any(_art):
                ax.fill_between(tw, _ylim[0], _ylim[1], where=_art,
                                color='darkorange', alpha=0.15, zorder=0)
            ax.axvline(0, color='k', linestyle='--', linewidth=0.7, alpha=0.6)
            ax.axvspan(_rv_cfg.BASELINE_TMIN, _rv_cfg.BASELINE_TMAX, color='gold', alpha=0.12, zorder=0)
            ax.axvspan(_rv_cfg.SCORE_TMIN, _rv_cfg.SCORE_TMAX, color='green', alpha=0.08, zorder=0)
            bm = t.get('baseline_mean')
            if bm is not None and not np.isnan(bm):
                ax.axhline(bm, color='goldenrod', linewidth=0.7, linestyle=':', alpha=0.8)
            ax.set_ylim(_ylim)
            ax.set_title(title, fontsize=7, pad=2,
                         color='darkorange' if (was_cleaned and not t['rejected'])
                               else ('dimgray' if t['rejected'] else 'black'))
            ax.tick_params(labelsize=5.5); ax.set_xlabel('s', fontsize=5)
            if i % N_COLS == 0: ax.set_ylabel('BPM', fontsize=6)

        for j in range(n, len(axes_flat)): axes_flat[j].axis('off')
        fig.legend(handles=[
            plt.Line2D([0],[0], color=config.NEG_COLOR, lw=1.5, label='Negative'),
            plt.Line2D([0],[0], color=config.NEU_COLOR, lw=1.5, label='Neutral'),
            plt.Line2D([0],[0], color=config.REJ_COLOR, lw=1.5, label='Rejected'),
            plt.patches.Patch(color='darkorange', alpha=0.4, label='~cleaned'),
            plt.patches.Patch(color='gold', alpha=0.5, label='Baseline'),
            plt.patches.Patch(color='green', alpha=0.4, label='Score'),
        ], fontsize=8, loc='lower right', ncol=6, framealpha=0.9)
        plt.tight_layout(rect=[0, 0.04, 1, 0.96]); plt.show()

In [ ]:
if SHOW_SUBJECT_PLOTS:
    # ── Subject average timecourse (Eve | Mor) ────────────────────────────────
    for subj, sessions in sorted(trials_data.items()):
        fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
        fig.suptitle(f'{subj} — Average HR Timecourse', fontsize=12, fontweight='bold')

        for ax, sess_key in zip(axes, ['eve', 'mor']):
            trials = sessions.get(sess_key, [])
            neg_e, neu_e, times = [], [], None
            for t in trials:
                if t['rejected'] or t['epoch_anal'] is None: continue
                ep = _pct(t['epoch_anal'], t['baseline_mean'])
                (neg_e if t['label'] == 1 else neu_e).append(ep)
                if times is None: times = t['times_anal']
            if times is None:
                ax.set_title(f"{config.SESSION_MAP[sess_key]['label']} — no data"); continue
            for eps, clr, lbl in [(neg_e, config.NEG_COLOR, 'Negative'),
                                   (neu_e, config.NEU_COLOR, 'Neutral')]:
                if not eps: continue
                arr = np.array(eps)
                avg = np.mean(arr, axis=0)
                sem = stats.sem(arr, axis=0, nan_policy='omit') if len(arr)>1 else np.zeros_like(avg)
                ax.plot(times, avg, color=clr, linewidth=2.2, label=f'{lbl} (n={len(eps)})')
                ax.fill_between(times, avg-sem, avg+sem, color=clr, alpha=0.2)
            ax.axvline(0, color='k', linestyle='--', linewidth=1.2)
            ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)
            ax.axvspan(config.SCORE_TMIN, config.SCORE_TMAX, color='green', alpha=0.07)
            ax.axvspan(config.BASELINE_TMIN, config.BASELINE_TMAX, color='gold', alpha=0.07)
            ax.set_title(config.SESSION_MAP[sess_key]['label'], fontsize=11, fontweight='bold')
            ax.set_xlabel('Time (s)'); ax.set_ylabel(YLABEL); ax.legend(fontsize=9)
        plt.tight_layout(); plt.show()

In [ ]:
if SHOW_SUBJECT_PLOTS:
    # ── Group average timecourse (Neg vs Neu, Eve | Mor panels) ──────────────
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
    fig.suptitle('Group Average HR Timecourse', fontsize=13, fontweight='bold')

    for ax, sess_key in zip(axes, ['eve', 'mor']):
        neg_e, neu_e, times = [], [], None
        n_neg_s = n_neu_s = 0
        for subj, sessions in trials_data.items():
            trials = sessions.get(sess_key, [])
            s_neg = [_pct(t['epoch_anal'], t['baseline_mean'])
                     for t in trials if not t['rejected'] and t['label']==1
                     and t['epoch_anal'] is not None]
            s_neu = [_pct(t['epoch_anal'], t['baseline_mean'])
                     for t in trials if not t['rejected'] and t['label']==2
                     and t['epoch_anal'] is not None]
            if s_neg: neg_e.append(np.mean(s_neg, axis=0)); n_neg_s += 1
            if s_neu: neu_e.append(np.mean(s_neu, axis=0)); n_neu_s += 1
            if times is None:
                times = next((t['times_anal'] for t in trials if t['times_anal'] is not None), None)
        if times is None:
            ax.set_title(f"{config.SESSION_MAP[sess_key]['label']} — no data"); continue
        for eps, clr, lbl, ns in [(neg_e, config.NEG_COLOR, 'Negative', n_neg_s),
                                   (neu_e, config.NEU_COLOR, 'Neutral',  n_neu_s)]:
            if not eps: continue
            arr = np.array(eps)
            avg = np.mean(arr, axis=0)
            sem = stats.sem(arr, axis=0, nan_policy='omit') if len(arr)>1 else np.zeros_like(avg)
            ax.plot(times, avg, color=clr, linewidth=2.2, label=f'{lbl} (n subj={ns})')
            ax.fill_between(times, avg-sem, avg+sem, color=clr, alpha=0.2)
        ax.axvline(0, color='k', linestyle='--', linewidth=1.2)
        ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)
        ax.axvspan(config.SCORE_TMIN, config.SCORE_TMAX, color='green', alpha=0.07)
        ax.axvspan(config.BASELINE_TMIN, config.BASELINE_TMAX, color='gold', alpha=0.07)
        ax.set_title(config.SESSION_MAP[sess_key]['label'], fontsize=12, fontweight='bold')
        ax.set_xlabel('Time (s)'); ax.set_ylabel(YLABEL); ax.legend(fontsize=9)
    plt.tight_layout(); plt.show()

In [ ]:
# ── Overall timecourse: Evening vs Morning ────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
times = None
for sess_key, clr, lbl in [('eve', config.EVE_COLOR, 'Evening'),
                             ('mor', config.MOR_COLOR, 'Morning')]:
    subj_avgs = []
    for subj, sessions in trials_data.items():
        trials = sessions.get(sess_key, [])
        accepted = [t for t in trials if not t['rejected'] and t['epoch_anal'] is not None]
        if not accepted: continue
        subj_avgs.append(np.mean([_pct(t['epoch_anal'], t['baseline_mean'])
                                   for t in accepted], axis=0))
        if times is None: times = accepted[0]['times_anal']
    if not subj_avgs or times is None: continue
    arr = np.array(subj_avgs)
    avg = np.mean(arr, axis=0)
    sem = stats.sem(arr, axis=0, nan_policy='omit') if len(arr)>1 else np.zeros_like(avg)
    ax.plot(times, avg, color=clr, linewidth=2.5, label=f'{lbl} (n={len(subj_avgs)})')
    ax.fill_between(times, avg-sem, avg+sem, color=clr, alpha=0.2)
ax.axvline(0, color='k', linestyle='--', linewidth=1.2)
ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)
ax.axvspan(config.SCORE_TMIN, config.SCORE_TMAX, color='green', alpha=0.07)
ax.axvspan(config.BASELINE_TMIN, config.BASELINE_TMAX, color='gold', alpha=0.07)
ax.set_xlabel('Time (s)', fontsize=11); ax.set_ylabel(YLABEL, fontsize=11)
ax.set_title('Overall Average HR: Evening vs Morning (±SEM)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
if SHOW_SUBJECT_PLOTS:
    # ── Group score boxplot (4 conditions) ───────────────────────────────────
    cond_keys = [('eve', 1, 'Eve-Neg', config.NEG_COLOR),
                 ('eve', 2, 'Eve-Neu', config.NEU_COLOR),
                 ('mor', 1, 'Mor-Neg', '#922B21'),
                 ('mor', 2, 'Mor-Neu', '#1A5276')]

    subj_means = {}
    for subj, sessions in trials_data.items():
        subj_means[subj] = {}
        for sk, cond, *_ in cond_keys:
            vals = [t['score'] for t in sessions.get(sk, [])
                    if not t['rejected'] and t['label'] == cond
                    and t['score'] is not None and not np.isnan(t['score'])]
            if vals:
                subj_means[subj][(sk, cond)] = np.mean(vals)

    valid = [s for s in subj_means if all((sk,c) in subj_means[s] for sk,c,*_ in cond_keys)]
    if len(valid) < 2:
        print('Not enough subjects')
    else:
        data_cols = [[subj_means[s][(sk,c)] for s in valid] for sk,c,*_ in cond_keys]
        n_trials  = [sum(1 for s in valid for t in trials_data[s].get(sk,[])
                         if not t['rejected'] and t['label']==c) for sk,c,*_ in cond_keys]
        fig, ax = plt.subplots(figsize=(10, 6))
        bp = ax.boxplot(data_cols, positions=[1,2,3,4], widths=0.45, patch_artist=True, showfliers=False)
        for patch,(sk,c,lbl,clr) in zip(bp['boxes'], cond_keys):
            patch.set_facecolor(clr); patch.set_alpha(0.65); patch.set_edgecolor('gray')
        for med in bp['medians']: med.set(color='black', linewidth=2)
        for s in valid:
            j = np.random.uniform(-0.07, 0.07)
            for (pn,pu),sk in [((1,2),'eve'),((3,4),'mor')]:
                vn = subj_means[s].get((sk,1)); vu = subj_means[s].get((sk,2))
                if vn is not None and vu is not None:
                    lc = config.UP_COLOR if vu>=vn else config.DOWN_COLOR
                    ax.plot([pn+j,pu+j],[vn,vu],color=lc,alpha=0.45,lw=1.2)
                    ax.scatter(pn+j,vn,color=cond_keys[0 if pn==1 else 2][3],edgecolors='k',s=45,zorder=4,lw=0.5)
                    ax.scatter(pu+j,vu,color=cond_keys[1 if pu==2 else 3][3],edgecolors='k',s=45,zorder=4,lw=0.5)
        for ca,cb,x1,x2 in [(data_cols[0],data_cols[1],1,2),(data_cols[2],data_cols[3],3,4)]:
            try:    _,p = stats.wilcoxon(ca,cb); tl='Wilcoxon'
            except: _,p = stats.ttest_rel(ca,cb); tl='Paired t'
            y_top = max(max(ca),max(cb))*1.08
            ax.set_ylim(top=max(ax.get_ylim()[1], y_top*1.25))
            _bracket(ax,x1,x2,y_top,p,tl)
        ax.set_xticks([1,2,3,4])
        ax.set_xticklabels([f'{lbl}\n(n={nt})' for (_,_,lbl,_),nt in zip(cond_keys,n_trials)])
        ax.set_ylabel(YLABEL, fontsize=11)
        ax.set_title(f'HR Score by Condition  (N={len(valid)} subjects)', fontweight='bold')
        ax.axvline(2.5, color='lightgray', linestyle='--', lw=1)
        plt.tight_layout(); plt.show()

In [ ]:
# ── Eve vs Mor — Subjective Negative, sound-included trials only ──────────────
eve_scores, mor_scores, valid_subjs = [], [], []

for subj, sessions in sorted(trials_data.items()):
    s = {}
    for sk in ['eve', 'mor']:
        vals = [t['score'] for t in sessions.get(sk, [])
                if not t['rejected']
                and t.get('subjective_label', t['label']) == 1
                and t['score'] is not None and not np.isnan(t['score'])]
        if vals: s[sk] = np.mean(vals)
    if 'eve' in s and 'mor' in s:
        eve_scores.append(s['eve']); mor_scores.append(s['mor']); valid_subjs.append(subj)

# IQR outlier removal on paired (Eve − Mor) difference
eve_scores, mor_scores, valid_subjs, n_removed = _iqr_filter_paired(
    eve_scores, mor_scores, valid_subjs, IQR_OUTLIER_MULTIPLIER)
if n_removed:
    print(f'IQR outlier removal: {n_removed} subject(s) removed (paired Eve−Mor difference)')

n = len(valid_subjs)
if n < 2:
    print('Not enough subjects')
else:
    try:    _, p = stats.wilcoxon(eve_scores, mor_scores); tl = 'Wilcoxon'
    except: _, p = stats.ttest_rel(eve_scores, mor_scores); tl = 'Paired t'
    fig, ax = plt.subplots(figsize=(6, 6))
    bp = ax.boxplot([eve_scores, mor_scores], positions=[1,2], widths=0.4,
                    patch_artist=True, showfliers=False)
    for patch, clr in zip(bp['boxes'], [config.EVE_COLOR, config.MOR_COLOR]):
        patch.set_facecolor(clr); patch.set_alpha(0.6); patch.set_edgecolor('gray')
    for med in bp['medians']: med.set(color='black', linewidth=2)
    for ev, mo in zip(eve_scores, mor_scores):
        j = np.random.uniform(-0.06, 0.06)
        lc = config.UP_COLOR if mo >= ev else config.DOWN_COLOR
        ax.plot([1+j,2+j],[ev,mo],color=lc,alpha=0.55,linewidth=1.2)
        ax.scatter(1+j,ev,color=config.EVE_COLOR,edgecolors='k',s=55,zorder=4,linewidths=0.5)
        ax.scatter(2+j,mo,color=config.MOR_COLOR,edgecolors='k',s=55,zorder=4,linewidths=0.5)
    h = (ax.get_ylim()[1]-ax.get_ylim()[0])*0.015
    y_top = max(max(eve_scores),max(mor_scores))*1.08
    ax.set_ylim(top=max(ax.get_ylim()[1], y_top*1.25))
    ax.plot([1,1,2,2],[y_top,y_top+h,y_top+h,y_top],lw=1.2,color='k')
    ax.text(1.5,y_top+h*1.5,f'{_sig(p)}  {tl}  p={p:.3f}',
            ha='center',va='bottom',fontsize=9,fontweight='bold')
    ax.set_xticks([1,2])
    ax.set_xticklabels([f'Evening\n(n={n})',f'Morning\n(n={n})'],fontsize=11)
    ax.set_ylabel(YLABEL, fontsize=11)
    ax.set_title(f'HR Score — Subjective Negative\nEvening vs Morning  (N={n} subjects)',
                 fontweight='bold')
    ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)
    plt.tight_layout(); plt.show()
    print(f'Eve mean={np.mean(eve_scores):.2f}%  Mor mean={np.mean(mor_scores):.2f}%  p={p:.3f}')